In [ ]:
# ============================================
# 1) Imports
# ============================================
import mlflow.pyfunc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Para gráficos mais bonitos
sns.set(style="whitegrid", palette="viridis")


In [ ]:
# ============================================
# 2) Configurações manuais
# ============================================

# Informe aqui as versões que você quer comparar
model_v1 = "2026.04.25.1455"   # modelo antigo
model_v2 = "2026.04.26.2146"   # modelo novo

# Caminhos dos modelos
path_v1 = f"models/dev/model_{model_v1}"
path_v2 = f"models/dev/model_{model_v2}"

print("Modelo 1:", path_v1)
print("Modelo 2:", path_v2)


In [ ]:
# ============================================
# 3) Carregar modelos
# ============================================
model1 = mlflow.pyfunc.load_model(path_v1)
model2 = mlflow.pyfunc.load_model(path_v2)

print("✔ Modelos carregados com sucesso")


In [ ]:
# ============================================
# 4) Amostragem aleatória DIRETO DO CSV
# ============================================

import pandas as pd
import numpy as np

csv_path = "data/processed/itbi_features_minimal.csv"
sample_size = 5000

# Colunas realmente usadas no modelo
cols = [
    "bairro",
    "area_do_terreno_m2",
    "valor_m2",
    "ano_mes",
    "media_valor_cep",
    "valor_venal_de_referencia"
]

# Conta total de linhas do arquivo (menos o header)
total_lines = sum(1 for _ in open(csv_path)) - 1

# Seleciona linhas aleatórias para pular
skip = sorted(np.random.choice(
    np.arange(1, total_lines + 1),
    size=total_lines - sample_size,
    replace=False
))

# Lê apenas a amostra e apenas as colunas necessárias
df_sample = pd.read_csv(
    csv_path,
    sep=";",
    skiprows=skip,
    usecols=cols
)

print("✔ Amostra carregada:", df_sample.shape)
df_sample.head()


In [ ]:
# ============================================
# 5) Previsões dos dois modelos
# ============================================

pred1 = model1.predict(X)
pred2 = model2.predict(X)

df_results = pd.DataFrame({
    "y_true": y_true,
    "pred_v1": pred1,
    "pred_v2": pred2
})

df_results["diff"] = df_results["pred_v2"] - df_results["pred_v1"]
df_results["abs_diff"] = df_results["diff"].abs()

df_results.head()


In [ ]:
# ============================================
# 6) Métricas de comparação
# ============================================
from sklearn.metrics import mean_absolute_error, r2_score

mae_v1 = mean_absolute_error(y_true, pred1)
mae_v2 = mean_absolute_error(y_true, pred2)

r2_v1 = r2_score(y_true, pred1)
r2_v2 = r2_score(y_true, pred2)

print("===== MÉTRICAS =====")
print(f"Modelo {model_v1} → MAE: {mae_v1:,.2f} | R²: {r2_v1:.4f}")
print(f"Modelo {model_v2} → MAE: {mae_v2:,.2f} | R²: {r2_v2:.4f}")


In [ ]:
# ============================================
# 7) Gráfico de comparação de MAE e R²
# ============================================

metrics_df = pd.DataFrame({
    "Modelo": [model_v1, model_v2],
    "MAE": [mae_v1, mae_v2],
    "R2": [r2_v1, r2_v2]
})

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=metrics_df, x="Modelo", y="MAE", ax=ax[0])
ax[0].set_title("Comparação de MAE")

sns.barplot(data=metrics_df, x="Modelo", y="R2", ax=ax[1])
ax[1].set_title("Comparação de R²")

plt.show()


In [ ]:
# ============================================
# 8) Distribuição das diferenças
# ============================================

plt.figure(figsize=(10,5))
sns.histplot(df_results["diff"], bins=50, kde=True)
plt.title("Distribuição das diferenças (Modelo Novo - Modelo Antigo)")
plt.xlabel("Diferença de Previsão")
plt.show()


In [ ]:
# ============================================
# 9) Conclusão automática
# ============================================

print("\n===== CONCLUSÃO =====")

if mae_v2 < mae_v1:
    print("✔ O modelo NOVO tem MAE menor → MELHOR")
else:
    print("✖ O modelo NOVO tem MAE maior → PIOR")

if r2_v2 > r2_v1:
    print("✔ O modelo NOVO tem R² maior → MELHOR")
else:
    print("✖ O modelo NOVO tem R² menor → PIOR")

print("\nDiferença média de previsão:", df_results["diff"].mean())
print("Diferença média absoluta:", df_results["abs_diff"].mean())
